# ComfyUI — klein-9B Skin Enhancer (clean rebuild)

**Before running:**
1. Runtime → Change runtime type → **GPU (A100/L4)**.
2. Add your **HF_TOKEN** in the 🔑 Secrets tab (and toggle notebook access ON).
3. Accept the klein-9B license once at https://huggingface.co/black-forest-labs/FLUX.2-klein-9B → 'Agree and access'.

Then run cells **1 → 4** top to bottom. Everything installs into `/content/ComfyUI` (local disk).
On a fresh runtime, just re-run all four — no manual file copying.

_Cell 4 stays running and embeds the ComfyUI UI inline — scroll down inside its output to use it._

In [1]:
# CELL 1 — Install ComfyUI + Manager + dependencies
import os
COMFY = '/content/ComfyUI'
if not os.path.isdir(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY}
%cd {COMFY}
!pip install -q -r requirements.txt           # installs comfy_aimdo etc. (fixes the old ModuleNotFound)
if not os.path.isdir(f'{COMFY}/custom_nodes/comfyui-manager'):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager {COMFY}/custom_nodes/comfyui-manager
print('✅ ComfyUI + Manager ready')

Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 42963, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 42963 (delta 23), reused 13 (delta 11), pack-reused 42923 (from 2)
Receiving objects: 100% (42963/42963), 83.80 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (29149/29149), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 143.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.9/418.9 kB 3

In [2]:
# CELL 2 — Custom nodes the workflow needs
import os, subprocess
CN = '/content/ComfyUI/custom_nodes'
repos = {
    # --- nodes REQUIRED by ai_skin_api.json ---
    'ComfyUI_Comfyroll_CustomNodes': 'https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes',  # CR Prompt Text, CR Text Concatenate (nodes 64,75)
    'ComfyUI-Easy-Use':              'https://github.com/yolain/ComfyUI-Easy-Use',                # easy imageColorMatch (node 83)
    'ComfyUI_EasySize':              'https://github.com/balu112121/ComfyUI_EasySize',            # EasySizeSimpleImage / 简单图像尺寸 (node 43)
    # --- kept from before (not used by this workflow, harmless) ---
    'ComfyUI_essentials':            'https://github.com/cubiq/ComfyUI_essentials',
    'comfyui_face_parsing':          'https://github.com/Ryuukeisyou/comfyui_face_parsing',
    'ComfyUI_LayerStyle':            'https://github.com/chflame163/ComfyUI_LayerStyle',
    'ComfyUI-Impact-Pack':           'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
    'ComfyUI-KJNodes':               'https://github.com/kijai/ComfyUI-KJNodes',
    'ComfyUI-SeedVR2_VideoUpscaler': 'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler',
}
for folder, url in repos.items():
    p = os.path.join(CN, folder)
    if os.path.isdir(p): print('⏭️', folder)
    else: print('⬇️', folder); subprocess.run(['git','clone','--depth','1',url,p], check=False)
for folder in repos:
    req = os.path.join(CN, folder, 'requirements.txt')
    if os.path.exists(req): subprocess.run(['pip','install','-q','-r',req], check=False)
print('✅ custom nodes ready (face_parsing + seedvr2 download their model weights on first run)')


⬇️ ComfyUI_Comfyroll_CustomNodes
⬇️ ComfyUI-Easy-Use
⬇️ ComfyUI_EasySize
⬇️ ComfyUI_essentials
⬇️ comfyui_face_parsing
⬇️ ComfyUI_LayerStyle
⬇️ ComfyUI-Impact-Pack
⬇️ ComfyUI-KJNodes
⬇️ ComfyUI-SeedVR2_VideoUpscaler
✅ custom nodes ready (face_parsing + seedvr2 download their model weights on first run)


In [3]:
# CELL 3 — Models: klein-9B + qwen_3_8b + VAE + LoRAs the workflow needs
import os, shutil
from huggingface_hub import hf_hub_download, list_repo_files
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
C = '/content/ComfyUI'
for d in ['diffusion_models','text_encoders','vae','loras']:
    os.makedirs(f'{C}/models/{d}', exist_ok=True)

def pull(repo, fname, dest, out_name=None, token=None):
    p = hf_hub_download(repo, fname, local_dir=dest, token=token)
    flat = os.path.join(dest, out_name or os.path.basename(fname))
    if p != flat: shutil.move(p, flat)
    # remove any leftover split_files/ subdir
    sub = os.path.join(dest, fname.split('/')[0])
    if '/' in fname and os.path.isdir(sub): shutil.rmtree(sub, ignore_errors=True)
    return flat

# klein-9B diffusion model (GATED — needs HF_TOKEN + accepted license)  -> workflow node 13
pull('black-forest-labs/FLUX.2-klein-9B', 'flux-2-klein-9b.safetensors', f'{C}/models/diffusion_models', token=HF_TOKEN)
# qwen_3_8b text encoder  -> workflow node 14
pull('Comfy-Org/flux2-klein-9B', 'split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors', f'{C}/models/text_encoders', token=HF_TOKEN)
# VAE — workflow node 10 (VAELoader) expects the filename 'flux2-vae-9b.safetensors'
pull('Comfy-Org/flux2-dev', 'split_files/vae/flux2-vae.safetensors', f'{C}/models/vae', out_name='flux2-vae-9b.safetensors')

# === LoRAs the uploaded workflow (ai_skin_api.json) references BY EXACT NAME ===
#   node 38 -> flux-2-klein-NSFW.safetensors      node 82 -> Lenovo-UltraReal.safetensors
def grab_lora_file(repo, fname, out_name, revision=None, token=None):
    src = hf_hub_download(repo, fname, local_dir=f'{C}/models/loras', revision=revision, token=token)
    dst = f'{C}/models/loras/{out_name}'
    if src != dst: shutil.move(src, dst)
    print('✅ LoRA:', out_name)

# NSFW LoRA — native Flux.2-klein-9B, pinned to the exact commit you provided
grab_lora_file('msrcam/Flux.2_Klein_9B_LoRas', 'flux2klein_nsfw.safetensors',
               'flux-2-klein-NSFW.safetensors',
               revision='9c8c61dfa7d1314b6d04f946595dcb5865359e02')
# Lenovo UltraReal (Flux2) — HF mirror of the Civitai model (no token needed)
grab_lora_file('Danrisi/Lenovo_UltraReal_Flux2', 'lenovo_flux2.safetensors', 'Lenovo-UltraReal.safetensors')

# --- extra klein LoRAs kept from before (NOT used by this workflow, harmless to keep) ---
def grab_lora(repo, out_name):
    sf = [f for f in list_repo_files(repo) if f.endswith('.safetensors')][0]
    src = hf_hub_download(repo, sf, local_dir=f'{C}/models/loras')
    dst = f'{C}/models/loras/{out_name}'
    if src != dst: shutil.move(src, dst)
    print('✅ LoRA:', out_name)
grab_lora('dx8152/Flux2-Klein-9B-Enhanced-Details', 'klein-enhanced-details.safetensors')
grab_lora('linoyts/Flux2-Klein-Delight-LoRA',       'klein-delight.safetensors')

!ls -lh {C}/models/diffusion_models {C}/models/text_encoders {C}/models/vae {C}/models/loras


flux-2-klein-9b.safetensors:   0%|          | 0.00/18.2G [00:00<?, ?B/s]

split_files/text_encoders/qwen_3_8b_fp8m(…):   0%|          | 0.00/8.66G [00:00<?, ?B/s]

split_files/vae/flux2-vae.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

flux2klein_nsfw.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

✅ LoRA: flux-2-klein-NSFW.safetensors


...

✅ LoRA: Lenovo-UltraReal.safetensors


realistic.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

✅ LoRA: klein-enhanced-details.safetensors


checkpoint-1000/pytorch_lora_weights.saf(…):   0%|          | 0.00/33.6M [00:00<?, ?B/s]

✅ LoRA: klein-delight.safetensors
/content/ComfyUI/models/diffusion_models:
total 17G
-rw-r--r-- 1 root root 17G Jul  9 21:04 flux-2-klein-9b.safetensors
-rw-r--r-- 1 root root   0 Jul  9 21:00 put_diffusion_model_files_here

/content/ComfyUI/models/loras:
total 693M
drwxr-xr-x 2 root root 4.0K Jul  9 21:05 checkpoint-1000
-rw-r--r-- 1 root root 159M Jul  9 21:05 flux-2-klein-NSFW.safetensors
-rw-r--r-- 1 root root  33M Jul  9 21:05 klein-delight.safetensors
-rw-r--r-- 1 root root 317M Jul  9 21:05 klein-enhanced-details.safetensors
-rw-r--r-- 1 root root 187M Jul  9 21:05 Lenovo-UltraReal.safetensors
-rw-r--r-- 1 root root    0 Jul  9 21:00 put_loras_here

/content/ComfyUI/models/text_encoders:
total 8.1G
-rw-r--r-- 1 root root    0 Jul  9 21:00 put_text_encoder_files_here
-rw-r--r-- 1 root root 8.1G Jul  9 21:05 qwen_3_8b_fp8mixed.safetensors

/content/ComfyUI/models/vae:
total 321M
-rw-r--r-- 1 root root 321M Jul  9 21:05 flux2-vae-9b.safetensors
-rw-r--r-- 1 root root    0 Jul  9 2

In [ ]:
# CELL 4 — Launch ComfyUI (embedded inline). KEEP RUNNING. Scroll down in this cell's output for the UI.
import threading, time, socket
%cd /content/ComfyUI
def serve(port):
    while True:
        time.sleep(0.5)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(('127.0.0.1', port)) == 0: s.close(); break
        s.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1224)
threading.Thread(target=serve, daemon=True, args=(8188,)).start()
!python main.py --dont-print-server --enable-cors-header

/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-07-09 21:05:32.277
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
[INFO] 
Prestartup times for custom nodes:
[INFO]    0.0 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Easy-Use
[INFO]    6.2 seconds: /content/ComfyUI/custom_nodes/comfyui

<IPython.core.display.Javascript object>

FETCH ComfyRegistry Data: 30/160
FETCH ComfyRegistry Data: 35/160
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/clipspace.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/groupNode.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui/components/buttonGroup.js. This is likely caused by a custom node extension usin